# Quaks News Analyst

This notebook demonstrates usage of the `quaks_news_analyst` agent to generate news insights. The workflow involves multiple agents, including a coordinator, an aggregator, and a reporter, each contributing to the final output.

The Agent has two distinct use cases based on given input:

1- Batch ETL: In this use case, the agent is responsible for extracting, transforming, and loading (ETL) news data in batches. The agent processes large volumes of news articles, extracts relevant information, transforms it into a structured format, and loads it into a database or data warehouse for further analysis. This use case is activated by sending the message content "BATCH_ETL" to the agent.

2- QA: In this use case, the agent is designed to answer questions related to news insights. Users can ask specific questions about news topics, trends, or events, and the agent will provide relevant insights based on its analysis of news data. This use case is activated by sending a message content that contains a question related to news insights.



In [ ]:
%%capture
import os
import nest_asyncio
from IPython.display import Markdown, HTML, display
from dotenv import load_dotenv
from notebooks import experiment_utils
from app.core.container import Container
from app.interface.api.messages.schema import MessageRequest

os.chdir("..")
load_dotenv()
nest_asyncio.apply()

# start dependency injection container
container = Container()
container.init_resources()
container.wire(modules=[__name__])

# get checkpointer instance
graph_persistence_factory = container.graph_persistence_factory()
checkpointer = graph_persistence_factory.build_checkpoint_saver()

In [ ]:
# Create Workflow
xai_agent = experiment_utils.create_xai_agent(
    agent_type="quaks_news_analyst", llm_tag="grok-4-1-fast-non-reasoning", api_key=os.getenv("XAI_API_KEY")
)
xai_news_analyst = container.quaks_news_analyst_agent()
xai_workflow_builder = xai_news_analyst.get_workflow_builder(xai_agent["id"])
xai_workflow = xai_workflow_builder.compile(checkpointer=checkpointer)

### Batch ETL Use Case

In [ ]:
%%capture

message = MessageRequest(
    message_role="human",
    message_content="BATCH_ETL",
    agent_id=xai_agent["id"],
)

inputs = xai_news_analyst.get_input_params(message, schema="public")
config = xai_news_analyst.get_config(xai_agent["id"])
result = xai_workflow.invoke(inputs, config)
ai_message_content, workflow_state = xai_news_analyst.format_response(result)

In [ ]:
display(HTML(ai_message_content))

### QA Use Case

In [ ]:
%%capture

message = MessageRequest(
    message_role="human",
    message_content="Can you share the most important take of today's news?",
    agent_id=xai_agent["id"],
)

inputs = xai_news_analyst.get_input_params(message, schema="public")
config = xai_news_analyst.get_config(xai_agent["id"])
result = xai_workflow.invoke(inputs, config)
ai_message_content, workflow_state = xai_news_analyst.format_response(result)

In [ ]:
display(Markdown(ai_message_content))


---

## Internals:

This section dives into the internals of the workflow, showing the graph structure and the system prompts for each agent involved in the workflow.


In [ ]:
experiment_utils.print_graph(xai_workflow)

In [ ]:
print(f"### Coordinator system prompt\n\n{result['coordinator_system_prompt']}\n")

In [ ]:
print(f"### Aggregator system prompt\n\n{result['aggregator_system_prompt']}\n")

In [ ]:
print(f"### Reporter system prompt\n\n{result['reporter_system_prompt']}\n")
